# agentic-rl-wordle — 多輪 GRPO 訓練（Colab A100）

流程：①參數 → ②Drive+bundle+安裝 → ③HF_TOKEN → ④pytest 煙霧 → ⑤(SMOKE專屬) spike →
⑥訓練＋（SMOKE專屬）自動續跑演練＋等待完成＋gate 判定＋（FULL專屬）評測+push＋釋放機器。

**這份 notebook 可以直接「全部執行」，不需要手動盯著重跑任何 cell。**

- **SMOKE_TEST=True**（預設）：跑猜數字煙霧 + 自動做 M2.4 續跑演練（會主動 kill 訓練行程模擬斷線、
  用 `--resume auto` 重啟、驗證 checkpoint 真的接續），全部跑完後自動判定 M2.3 gate（reward/win_rate
  是否上升）。**cell ⑥ 預期總耗時約 40–70 分鐘**（含等 checkpoint 出現、kill、重啟驗證、剩餘訓練時間），
  不是原本估的 20–30 分鐘——因為多做了續跑演練。
- **SMOKE_TEST=False**：跳過 spike 與續跑演練，直接跑正式 Wordle 訓練到 `--max-hours` 預算，
  結束後自動跑評測、push LoRA+merged 到 HF、最後釋放機器。
- 斷線 SOP（真的斷線，不是自動演練那種）：重新執行 ①②③⑥（RESUME="auto" 會從 Drive 最新 checkpoint 接續）
- 背景執行請開：執行階段 → 背景執行；**cell ⑥ 結尾一定會呼叫 `runtime.unassign()` 釋放機器**
  （用 try/finally 包住，就算中間任何一步出錯也會執行，不會留著空燒）

In [ ]:
# ===== ① 參數（唯一需要手改的 cell）=====
SMOKE_TEST = True                 # True: 猜數字煙霧；False: Wordle 正式訓練
RUN_NAME = "wordle-grpo-v1"       # Drive 上 checkpoint/log 的資料夾名
HF_USERNAME = "steven0226"
REWARD_PRESET = "shaped"          # shaped | binary（HF 官方發現 binary 對 Wordle 更穩，留作 A/B）
MAX_HOURS = 8.0                   # 正式訓練牆鐘預算，到點自動存檔停訓
RESUME = "auto"

DRIVE_BASE = "/content/drive/MyDrive/agentic-rl-wordle"

In [ ]:
# ===== ② Drive + 原始碼 bundle + 依賴安裝 =====
from google.colab import drive
drive.mount('/content/drive')

import pathlib
BUNDLE = f"{DRIVE_BASE}/wordle_rl_bundle.zip"
assert pathlib.Path(BUNDLE).exists(), f"先在本機執行 scripts/make_colab_bundle.py 並把 zip 上傳到 {BUNDLE}"

!rm -rf /content/agentic-rl-wordle && mkdir -p /content/agentic-rl-wordle
!unzip -q -o "$BUNDLE" -d /content/agentic-rl-wordle
%cd /content/agentic-rl-wordle

# ⚠️ 版本 pin 原則：torch 用 Colab 內建 CUDA build（絕不覆蓋，專案 2 教訓）；
#    trl/vllm/peft 精確 pin 於 requirements-colab.txt（M2.1 spike 已驗證過此組合可行）
!pip install -q -e . pytest
!pip install -q -r requirements-colab.txt

import torch, transformers, trl, vllm
print("torch", torch.__version__, "| trl", trl.__version__,
      "| transformers", transformers.__version__, "| vllm", vllm.__version__)
assert trl.__version__.startswith("1.8"), "trl 版本漂移——對照 requirements-colab.txt 與 docs/decision.md"

# 單字表 fetch-at-setup（出處與 sha256 → data/SOURCE.json；計數斷言 2315/10657/12972）
!python scripts/fetch_words.py

In [ ]:
# ===== ③ HF_TOKEN（Colab Secrets 需事先設定）=====
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("HF_TOKEN OK")

In [ ]:
# ===== ④ 煙霧驗證（<60 秒；不全綠就不要燒 GPU）=====
!python -m pytest tests -q

In [ ]:
# ===== ⑤（SMOKE 專屬）M2.1 spike：新環境首跑必執行——驗證 TRL 接觸面、記錄版本三元組 =====
# 過 → 把印出的版本回填 requirements-colab.txt、結論寫 docs/decision.md
# 不過 → 依錯誤訊息修 rollout.py / train.py / requirements-colab.txt；60 分鐘 timebox，超時轉 ART 備援
#
# ⚠️ 用 subprocess.run + assert，不用 `!python`：spike 失敗時「全部執行」必須在這裡硬停，
#    不能悶頭跑進昂貴的 cell ⑥（訓練）去空燒 GPU
#    （`!python ...` 這種 shell 魔法指令即使子行程回傳非 0，Colab 也不會讓 cell 失敗）。
# ⚠️ 一定要 capture_output=True 再自己 print 出來：子行程直接繼承 stdout 時，
#    Colab 不保證會把它顯示在 cell 輸出裡（實測只看得到後面 assert 的 traceback，
#    看不到 spike 腳本自己印的診斷內容）——用 Python 的 print() 才保證看得到。
import subprocess
import sys

if SMOKE_TEST:
    result = subprocess.run(
        [sys.executable, "scripts/spike_trl.py"], capture_output=True, text=True
    )
    print(result.stdout)
    if result.stderr:
        print("--- stderr ---")
        print(result.stderr)
    assert result.returncode == 0, (
        f"M2.1 spike 失敗（returncode={result.returncode}）——修好前不要往下跑 cell ⑥"
    )

In [ ]:
# ===== ⑥ 訓練 →（SMOKE）自動續跑演練 → 等待完成 → gate 判定 →（FULL）評測+push → 釋放機器 =====
# 全部包在一個 try/finally 裡：不管中間哪一步失敗，最後都會呼叫 runtime.unassign()，
# 不會留著 GPU 空燒。SMOKE_TEST=True 時會多做 M2.4 續跑演練（真的 SIGKILL 訓練行程模擬斷線，
# 再用 --resume auto 重啟，比對 checkpoint 前後的 global_step 確認真的接續而非從頭重跑）。

import json
import pathlib
import shutil
import subprocess
import sys
import time

CKPT_DIR = f"{DRIVE_BASE}/runs/{RUN_NAME}" + ("-smoke" if SMOKE_TEST else "")
pathlib.Path(CKPT_DIR).mkdir(parents=True, exist_ok=True)
LOG_PATH = f"{CKPT_DIR}/train.log"
METRICS_PATH = pathlib.Path(CKPT_DIR) / "metrics.jsonl"


def launch_training():
    cmd = [sys.executable, "-m", "wordle_rl.train",
           "--preset", "smoke" if SMOKE_TEST else "full",
           "--reward", REWARD_PRESET,
           "--output-dir", CKPT_DIR,
           "--resume", RESUME,
           "--max-hours", str(0.5 if SMOKE_TEST else MAX_HOURS)]
    print(">>>", " ".join(cmd), flush=True)
    log_f = open(LOG_PATH, "ab")
    p = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT)
    print("PID:", p.pid, "| log:", LOG_PATH, flush=True)
    return p


def read_metrics():
    if not METRICS_PATH.exists():
        return []
    return [json.loads(l) for l in METRICS_PATH.read_text().splitlines() if l.strip()]


def tail_log(n=30):
    if not pathlib.Path(LOG_PATH).exists():
        return "(尚無 log)"
    return "\n".join(pathlib.Path(LOG_PATH).read_text(errors="ignore").splitlines()[-n:])


def latest_checkpoint_step():
    ckpts = sorted(
        pathlib.Path(CKPT_DIR).glob("checkpoint-*"),
        key=lambda p: int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else -1,
    )
    for p in reversed(ckpts):
        sp = p / "trainer_state.json"
        if sp.exists():
            try:
                return int(json.loads(sp.read_text())["global_step"])
            except (json.JSONDecodeError, KeyError):
                continue
    return None


resume_drill_result = "SKIPPED（FULL 模式不做 M2.4 演練）"

try:
    proc = launch_training()

    # ---------- M2.4 續跑演練（只在 SMOKE_TEST 做）----------
    if SMOKE_TEST:
        print("\n=== M2.4 續跑演練：等待第一個 checkpoint 出現（最長 20 分鐘）===", flush=True)
        deadline = time.monotonic() + 20 * 60
        ckpt_step = None
        while time.monotonic() < deadline:
            ckpt_step = latest_checkpoint_step()
            if ckpt_step is not None:
                break
            time.sleep(20)

        if ckpt_step is None:
            resume_drill_result = "FAIL（20 分鐘內沒出現 checkpoint，train.log 尾端如下）"
            print("❌", resume_drill_result, "\n", tail_log(), flush=True)
        else:
            n_before = len(read_metrics())
            print(f"✓ checkpoint 出現，global_step={ckpt_step}；SIGKILL 模擬斷線 …", flush=True)
            proc.kill()
            try:
                proc.wait(timeout=30)
            except subprocess.TimeoutExpired:
                print("⚠ 行程 30 秒內未結束（繼續往下走）", flush=True)
            time.sleep(5)
            print("以 --resume auto 重新啟動 …", flush=True)
            proc = launch_training()

            print("等待重啟後寫入新的 metrics 紀錄（最長 10 分鐘）…", flush=True)
            deadline = time.monotonic() + 10 * 60
            resumed_first_step = None
            while time.monotonic() < deadline:
                rows = read_metrics()
                if len(rows) > n_before:
                    resumed_first_step = rows[n_before]["step"]
                    break
                time.sleep(15)

            if resumed_first_step is None:
                resume_drill_result = "FAIL（重啟後 10 分鐘內沒有新的 metrics 紀錄）"
            elif resumed_first_step <= 2 and ckpt_step > 5:
                resume_drill_result = (
                    f"FAIL（重啟後第一步 step={resumed_first_step}，"
                    f"疑似從頭重跑而非接續 checkpoint step={ckpt_step}）"
                )
            else:
                resume_drill_result = (
                    f"PASS（checkpoint step={ckpt_step} → 重啟後接續 step={resumed_first_step}）"
                )
            print(("✅ " if resume_drill_result.startswith("PASS") else "❌ ") + resume_drill_result,
                  flush=True)

    # ---------- 等待訓練跑到 --max-hours / --max-steps 自然結束 ----------
    print("\n=== 等待訓練程序結束（受 --max-hours 內部約束，不會無限跑）===", flush=True)
    rc = proc.wait()
    print("訓練程序結束 returncode =", rc, flush=True)

    # ---------- M2.3 gate 自動判定（reward/win_rate 是否有上升）----------
    rows = read_metrics()
    print(f"\n=== M2.3 gate（{'SMOKE' if SMOKE_TEST else 'FULL'}，共 {len(rows)} 筆 metrics 紀錄）===",
          flush=True)
    if len(rows) >= 15:
        early = rows[:10]
        late = rows[-20:] if len(rows) >= 20 else rows[-max(1, len(rows) // 3):]

        def mean(key, subset):
            vals = [r[key] for r in subset if r.get(key) is not None]
            return sum(vals) / len(vals) if vals else None

        early_r, late_r = mean("reward/mean", early), mean("reward/mean", late)
        early_wr, late_wr = mean("rollout/win_rate", early), mean("rollout/win_rate", late)
        print(f"reward/mean   前段={early_r}  後段={late_r}")
        print(f"win_rate      前段={early_wr}  後段={late_wr}")
        gate_pass = (
            late_r is not None and early_r is not None and late_r > early_r
            and late_wr is not None and early_wr is not None and (late_wr - early_wr) >= 0.15
        )
        print("✅ M2.3 GATE PASS" if gate_pass
              else "❌ M2.3 GATE 未過（reward 或 win_rate 沒有明顯上升，人工檢視 samples/ 判斷原因）")
    else:
        print("樣本數不足以自動判定，人工檢視 metrics.jsonl / train.log")

    # ---------- 曲線圖（存到 Drive）----------
    if rows:
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
        for ax, key, title in [
            (axes[0], "reward/mean", "mean episode reward"),
            (axes[1], "rollout/win_rate", "rollout win rate"),
            (axes[2], "rollout/illegal_per_ep", "illegal turns / episode"),
        ]:
            pts = [(r["step"], r[key]) for r in rows if r.get(key) is not None]
            if pts:
                ax.plot(*zip(*pts))
            ax.set_title(title)
            ax.set_xlabel("step")
        plt.tight_layout()
        plt.savefig(f"{CKPT_DIR}/curves.png")
        plt.show()

    # ---------- FULL 專屬：評測 + push HF ----------
    if not SMOKE_TEST:
        adapter = f"{CKPT_DIR}/final"
        subprocess.run([sys.executable, "eval/run_eval.py",
                        "--adapter", adapter, "--backend", "vllm"], check=True)
        shutil.copytree("results", f"{CKPT_DIR}/results", dirs_exist_ok=True)
        subprocess.run([sys.executable, "scripts/push_model.py",
                        "--adapter", adapter,
                        "--repo", f"{HF_USERNAME}/qwen2.5-1.5b-wordle-grpo",
                        "--card", "docs/model_card.md"], check=True)
        print("HF push 完成 ✅", flush=True)

    print(f"\n=== 總結 ===\nM2.4 續跑演練：{resume_drill_result}", flush=True)
finally:
    from google.colab import runtime
    runtime.unassign()